# 02c · 2021 against 2025: did burning fall as fast as fire counts?

**Project:** Paddy Burnt-Area Intelligence (Satellite Mission Intelligence for India)
**Question for this notebook:** from 2021 to 2025, Suomi NPP fire detections in Sangrur fell by about 95%, from 9,076 to 482 (Notebook 01). Did the area Sentinel-2 sees burnt fall as much, or do fire counts now miss more of the burning?

**Why:** official stubble-burning monitoring counts fires (see the README), so falling counts suggest falling burning. If the burnt area fell much less than detections, the counts overstate the decline, and the agriculture department needs a burnt-area map to see what is happening.

**How it works**
1. Notebook 02 maps both seasons with the same rule, the same 5-day composites and the same district outline (pre-2021 boundary). Run it twice: once as it is (2025), and once with `SEASON_YEAR = 2021`.
2. This notebook reads both seasons' outputs and sets them side by side: harvested and burnt area, the timing, and fire detections.
3. **The missed share is recomputed on the same satellites for both seasons.** NOAA-21 wasn't flying in 2021 (it launched in November 2022), so Notebook 02's 2025 figures use three VIIRS satellites and its 2021 figures two. Here both seasons use Suomi NPP and NOAA-20, with Notebook 02's matching rule (within 375 m, between the last clear view and the burn). As a check, the notebook first reproduces Notebook 02's own 2025 figure from the saved files.
4. A map compares the burnt area of each 1 km square in the two seasons.

**Before you run it:** run Notebook 02 for both seasons on the whole district (`RUN_FULL = True`). Nothing is downloaded and no key is needed. It takes under a minute.

**What it produces**

| Output | Where |
|---|---|
| The two seasons side by side, with the change | `data/processed/season_comparison_sangrur.csv` |
| Burnt area and detections by 5-day window, both seasons | `data/processed/season_comparison_by_window_sangrur.csv` |
| Burnt area by band of the district, south to north | `data/processed/season_comparison_by_band_sangrur.csv` |
| Figures | `maps/notebook02c/` |

In [ ]:
# Colab: add libraries its default image may lack. On your own computer, install requirements.txt into the
# project's environment first; there this cell only checks that the install worked.
import importlib.util, subprocess, sys

try:
    import google.colab  # noqa: F401  (this module exists only on Colab)
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

NEEDED = ('rasterio', 'geopandas', 'matplotlib', 'pandas', 'scipy')
if ON_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio', 'geopandas'], check=True)
else:
    missing = [name for name in NEEDED if importlib.util.find_spec(name) is None]
    if missing:
        raise ModuleNotFoundError('Missing: ' + ', '.join(missing) + ". Install requirements.txt into the "
                                  "project's environment and re-run.")
    print('Libraries found.')

## 1. Settings

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import geopandas as gpd
import rasterio
from rasterio import features
from scipy.spatial import cKDTree
from IPython.display import display

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/India-Space-Mission-Intelligence')
elif os.environ.get('PROJECT_DIR'):
    PROJECT = Path(os.environ['PROJECT_DIR']).resolve()
else:
    # On your computer: walk up from the working folder to the project root (the folder with requirements.txt).
    here = Path.cwd().resolve()
    PROJECT = next((p for p in (here, *here.parents)
                    if (p / 'requirements.txt').exists() and (p / 'notebooks').is_dir()), here.parent)

PROCESSED = PROJECT / 'data' / 'processed'
FIGS = PROJECT / 'maps' / 'notebook02c'
FIGS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})
print('Project folder:', PROJECT)

AREA, AREA_LABEL = 'sangrur', 'Sangrur (pre-2021 boundary)'
EARLIER, LATER = 2021, 2025
VALIDATED_SEASON = 2025     # Notebook 02 keeps its downloads in sentinel2/sangrur/, other seasons' in sangrur_<year>/
SAME_SATELLITES = ['VIIRS_SNPP', 'VIIRS_NOAA20']    # flying in both seasons
STRICT_M = 375                                       # Notebook 02's strict matching distance (one VIIRS pixel)
WINDOW_DAYS = 5

## 2. Helper functions

`flagged_by` is Notebook 02's matching rule, copied unchanged: a burnt pixel is flagged when a detection lies within the distance, in any window from its last clear view before the burn to the window in which the burn shows.

In [ ]:
def load_season(year):
    # Notebook 02's outputs for one season, whole district.
    suffix = f'{AREA}_{year}'
    paths = {name: PROCESSED / f'{name}_{suffix}.{ext}' for name, ext in (
        ('s2_burn_summary', 'csv'), ('s2_burn_by_window', 'csv'), ('s2_burn_classes', 'tif'),
        ('firms_detections', 'csv'), ('s2_burn_grid_1km', 'geojson'))}
    missing = [p.name for p in paths.values() if not p.exists()]
    if missing:
        raise RuntimeError(f'Missing for {year}: {", ".join(missing)}. Run Notebook 02 for the whole district with '
                           f'SEASON_YEAR = {year} (and RUN_FULL = True) first.')
    summary = pd.read_csv(paths['s2_burn_summary']).set_index('metric')['value']
    with rasterio.open(paths['s2_burn_classes']) as src:
        classes, burn_band, _, prev_band = src.read().astype('int16')
        transform, crs = src.transform, src.crs
    det = pd.read_csv(paths['firms_detections'])
    det = gpd.GeoDataFrame(det, geometry=gpd.points_from_xy(det['longitude'], det['latitude']), crs=4326).to_crs(crs)
    return {'year': year, 'summary': summary, 'by_window': pd.read_csv(paths['s2_burn_by_window']),
            'classes': classes, 'burn_w': burn_band - 1, 'prev_w': prev_band - 1, 'transform': transform,
            'crs': crs, 'detections': det, 'grid': gpd.read_file(paths['s2_burn_grid_1km'])}


def flagged_by(det, radius_m, burn_w, prev_w, transform, shape):
    # Burnt pixels with a detection within radius_m, made between the last clear view before the burn and
    # the window in which the burn shows (Notebook 02, section 2).
    flagged = np.zeros(shape, bool)
    for w, grp in det.groupby('window'):
        near = features.rasterize(((geom.buffer(radius_m), 1) for geom in grp.geometry), out_shape=shape,
                                  transform=transform, fill=0, dtype='uint8').astype(bool)
        flagged |= (burn_w >= 0) & near & (prev_w <= w) & (w <= burn_w)
    return flagged


def missed_share(season, sensors, radius_m=STRICT_M):
    # Share of confirmed burnt area (classes 3 and 4) with no detection from `sensors` nearby, in time.
    confirmed = np.isin(season['classes'], [3, 4])
    det = season['detections'][season['detections']['sensor'].isin(sensors)]
    flags = flagged_by(det, radius_m, season['burn_w'], season['prev_w'], season['transform'], confirmed.shape)
    return 1 - (flags & confirmed).sum() / max(confirmed.sum(), 1)


def change(a, b):
    return (b - a) / a if a else np.nan


def look_dates(year):
    # Dates with Sentinel-2 data over the 10 km pilot box from 15 Oct to 24 Nov: the file names of Notebook 02's
    # cloud-mask check downloads. Downloads stay on the computer that ran Notebook 02, so this may find none.
    folder = PROJECT / 'data' / 'raw' / 'sentinel2' / (AREA if year == VALIDATED_SEASON else f'{AREA}_{year}')
    return sorted({p.stem[7:17] for p in (folder / 'pilot_scl_check').glob('openEO_*.tif')})

## 3. The two seasons side by side

Areas are in hectares. "Detections inside" counts detections whose centre falls inside the district during the composite period (5 Sep to 4 Dec). The missed share is of confirmed burns (classes 3 and 4): burns first seen from 19 Nov are unconfirmed in both seasons, as in Notebook 02.

The burnt share of land harvested in both seasons compares the same fields, so a change in where crops were grown can't affect it.

The clear share and the number of Sentinel-2 dates show how well each season was seen. More looks catch more burns before fields are ploughed. Sentinel-2C took over from Sentinel-2A on 21 January 2025 ([Copernicus](https://dataspace.copernicus.eu/node/2039)), and Sentinel-2A stayed on in an extended campaign from 13 March 2025 ([Copernicus](https://dataspace.copernicus.eu/news/2025-2-28-sentinel-2a-extended-campaign-starting-march-13-2025)). So 2025 may have had more looks than 2021, when Sentinel-2A and 2B were flying. The dates are counted over Notebook 02's 10 km pilot box, from its cloud-mask check downloads, so they are a sample of the district, and they are blank where those downloads aren't on this computer.

In [ ]:
seasons = {year: load_season(year) for year in (EARLIER, LATER)}
same_grid = seasons[EARLIER]['transform'] == seasons[LATER]['transform']
if same_grid:
    # Like for like: the land harvested in both seasons.
    harvested_both = np.isin(seasons[EARLIER]['classes'], [2, 3, 4, 5]) & \
        np.isin(seasons[LATER]['classes'], [2, 3, 4, 5])
    burnt_both = {year: np.isin(s['classes'], [3, 4]) & harvested_both for year, s in seasons.items()}
else:
    print('Note: the two seasons were mapped on slightly different grids. Area figures still compare, but the '
          'like-for-like share is left blank and the 1 km map in section 5 is matched by square centres.')

# Check: reproduce Notebook 02's own 2025 missed share (all VIIRS satellites) from the saved files.
all_viirs = [s for s in seasons[LATER]['detections']['sensor'].unique() if s.startswith('VIIRS')]
again = missed_share(seasons[LATER], all_viirs)
reported = float(seasons[LATER]['summary']['missed_share_viirs_375m'])
print(f'Check: {LATER} missed share with all VIIRS satellites recomputed here {again:.4f}, '
      f'Notebook 02 {reported:.4f}.' + ('' if abs(again - reported) < 0.002 else ' WARNING: they differ.'))

rows = {}
for year, s in seasons.items():
    summ, det = s['summary'], s['detections']
    inside = det[det['inside']]
    burnt = float(summ['burnt_ha'])
    rows[year] = {
        'harvested area (ha)': float(summ['harvested_ha']),
        'burnt area, confirmed (ha)': burnt,
        'burnt area, including unconfirmed late burns (ha)': burnt + float(summ['burnt_unconfirmed_ha']),
        'burnt share of harvested area': float(summ['burnt_share_of_harvested']),
        'burnt share of land harvested in both seasons':
            burnt_both[year].sum() / max(harvested_both.sum(), 1) if same_grid else np.nan,
        'median days from harvest to burn': float(summ['median_days_harvest_to_burn']),
        'mean clear share of the district per window': float(summ['mean_clear_share']),
        'Sentinel-2 dates over the pilot box, 15 Oct to 24 Nov': len(look_dates(year)) or np.nan,
        'Suomi NPP detections inside': int((inside['sensor'] == 'VIIRS_SNPP').sum()),
        'Suomi NPP and NOAA-20 detections inside': int(inside['sensor'].isin(SAME_SATELLITES).sum()),
        'all VIIRS detections inside': int(inside['sensor'].str.startswith('VIIRS').sum()),
        'MODIS detections inside': int((inside['sensor'] == 'MODIS').sum()),
        'Suomi NPP detections per 1,000 ha burnt': (inside['sensor'] == 'VIIRS_SNPP').sum() / burnt * 1000
        if burnt else np.nan,
        'missed share as reported by Notebook 02 (VIIRS satellites flying that year, 375 m)':
            float(summ['missed_share_viirs_375m']),
        'missed share, Suomi NPP and NOAA-20 only (375 m)': missed_share(s, SAME_SATELLITES),
    }
table = pd.DataFrame(rows)
# Areas and counts change by a percentage; shares and days by a difference.
as_difference = table.index.str.contains('share') | table.index.str.startswith('median days')
table['relative change'] = np.where(as_difference, np.nan, (table[LATER] - table[EARLIER]) / table[EARLIER])
table['difference'] = np.where(as_difference, table[LATER] - table[EARLIER], np.nan)
display(table.round(3))

det_drop = change(*table.loc['Suomi NPP detections inside', [EARLIER, LATER]])
burn_drop = change(*table.loc['burnt area, confirmed (ha)', [EARLIER, LATER]])
miss = table.loc['missed share, Suomi NPP and NOAA-20 only (375 m)', [EARLIER, LATER]]
print(f'From {EARLIER} to {LATER}: Suomi NPP detections inside the district changed by {det_drop:+.0%}; the burnt '
      f'area Sentinel-2 maps changed by {burn_drop:+.0%}. With the same two satellites, the share of burnt area no '
      f'fire detection flagged went from {miss[EARLIER]:.0%} to {miss[LATER]:.0%}.')
if same_grid:
    px_ha = abs(seasons[LATER]['transform'].a * seasons[LATER]['transform'].e) / 10_000
    again = (burnt_both[EARLIER] & burnt_both[LATER]).sum()
    print(f'On the {harvested_both.sum() * px_ha:,.0f} ha harvested in both seasons, '
          f'{again / max(burnt_both[LATER].sum(), 1):.0%} of the area burnt in {LATER} had burnt in {EARLIER} too, '
          f'and {again / max(burnt_both[EARLIER].sum(), 1):.0%} of the area burnt in {EARLIER} burnt again in {LATER}.')

fig, ax = plt.subplots(figsize=(7, 3.6))
names = ['Suomi NPP detections inside', 'burnt area, confirmed (ha)', 'harvested area (ha)']
labels = ['fire detections (Suomi NPP)', 'burnt area (Sentinel-2)', 'harvested area (Sentinel-2)']
values = [table.loc[n, LATER] / table.loc[n, EARLIER] * 100 if table.loc[n, EARLIER] else np.nan for n in names]
ax.barh(labels[::-1], values[::-1], color=['#bab0ac', '#e45756', '#4c78a8'][::-1])
for i, v in enumerate(values[::-1]):
    ax.text(v + 1, i, f'{v:.0f}', va='center')
ax.axvline(100, color='k', linewidth=0.8)
ax.set_xlabel(f'{LATER} as a percentage of {EARLIER}')
ax.set_title(f'{AREA_LABEL}: {LATER} against {EARLIER}')
fig.savefig(FIGS / '01_change_2021_2025.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. When it burned

Burnt area (confirmed and unconfirmed) and Suomi NPP detections inside the district, by 5-day window, for both seasons. Each is drawn as a percentage of its season's total, so the timing compares even though the totals differ; the totals are in the legends, and the hectares and counts are in `season_comparison_by_window_sangrur.csv`. The windows start on the same calendar days each year (5 Sep, 10 Sep, ...).

In [ ]:
by_window = []
for year, s in seasons.items():
    bw = s['by_window'].copy()
    det = s['detections']
    snpp = det[det['inside'] & (det['sensor'] == 'VIIRS_SNPP')].groupby('window').size()
    bw['season'] = year
    bw['window'] = range(len(bw))
    bw['day'] = pd.to_datetime(bw['window_start']).dt.strftime('%d %b')
    bw['snpp_detections_inside'] = bw['window'].map(snpp).fillna(0).astype(int)
    by_window.append(bw[['season', 'window', 'day', 'harvested_ha', 'burnt_ha', 'snpp_detections_inside',
                         'clear_share_of_area', 'burns_confirmed']])
by_window = pd.concat(by_window, ignore_index=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
for year, colour in ((EARLIER, 'tab:gray'), (LATER, 'tab:red')):
    part = by_window[by_window['season'] == year]
    for ax, column, unit in ((ax1, 'burnt_ha', 'ha'), (ax2, 'snpp_detections_inside', 'detections')):
        total = part[column].sum()
        ax.plot(part['window'], part[column] / total * 100 if total else part[column], marker='o', color=colour,
                label=f'{year}: {total:,.0f} {unit} in all')
ax1.set_ylabel('burnt area first seen\n(% of the season)')
ax2.set_ylabel('Suomi NPP detections\n(% of the season)')
ax1.set_title(f'{AREA_LABEL}: when burning was seen, by 5-day window')
ax1.legend()
ax2.legend()
days = by_window[by_window['season'] == LATER].set_index('window')['day']
ax2.set_xticks(days.index[::2])
ax2.set_xticklabels(days.iloc[::2], rotation=45, fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / '02_timing.png', bbox_inches='tight', dpi=150)
plt.show()
for year in (EARLIER, LATER):
    part = by_window[by_window['season'] == year]
    half = part.loc[part['burnt_ha'].cumsum() >= part['burnt_ha'].sum() / 2, 'day'].iloc[0]
    print(f'{year}: half the burnt area first seen by the window starting {half}.')

## 5. Where it changed

Burnt area in each 1 km square, 2025 minus 2021: blue squares burnt less in 2025, red squares more. The left map counts confirmed burns only; the right map adds the burns first seen from 19 Nov. The two differ where one season burnt later than the other: the cut-off falls on the same calendar day in both seasons, so it removes more of a late season's burning.

The table splits the district into four bands with equal numbers of squares, from south to north.

In [ ]:
COLS = ['burnt_ha', 'burnt_unconfirmed_ha']
grids = {year: s['grid'][['cell', *COLS, 'geometry']].set_index('cell') for year, s in seasons.items()}
if not same_grid:
    # Different grids: give each earlier square the id of the later square whose centre lies within half a
    # square (500 m) of its own centre. Squares with no such partner keep an id of their own.
    centres = {year: g.to_crs(seasons[LATER]['crs']).centroid for year, g in grids.items()}
    dist, nearest = cKDTree(np.c_[centres[LATER].x, centres[LATER].y]).query(
        np.c_[centres[EARLIER].x, centres[EARLIER].y], distance_upper_bound=500)
    matched = np.isfinite(dist)
    later_ids = grids[LATER].index.to_numpy()[np.minimum(nearest, len(grids[LATER]) - 1)]
    grids[EARLIER].index = np.where(matched, later_ids, 'unmatched ' + grids[EARLIER].index.astype(str))
    print(f'{matched.mean():.0%} of the {EARLIER} squares matched a {LATER} square.')
both = grids[LATER][['geometry']]
for year, tag in ((EARLIER, 'earlier'), (LATER, 'later')):
    both = both.join(grids[year][COLS].add_suffix(f'_{tag}'), how='outer')
both['geometry'] = both['geometry'].fillna(grids[EARLIER]['geometry'])
numbers = [f'{c}_{tag}' for c in COLS for tag in ('earlier', 'later')]
both[numbers] = both[numbers].fillna(0)
both = gpd.GeoDataFrame(both, geometry='geometry', crs=4326)
both['change_ha'] = both['burnt_ha_later'] - both['burnt_ha_earlier']
both['change_all_ha'] = both['change_ha'] + both['burnt_unconfirmed_ha_later'] - both['burnt_unconfirmed_ha_earlier']
limit = max(both[['change_ha', 'change_all_ha']].abs().stack().quantile(0.98), 1)
norm = TwoSlopeNorm(0, -limit, limit)
fig, axes = plt.subplots(1, 2, figsize=(12, 7.5))
for ax, column, title in zip(axes, ('change_ha', 'change_all_ha'),
                             ('confirmed burns', 'including burns first seen from 19 Nov')):
    both.plot(column=column, cmap='RdBu_r', norm=norm, ax=ax)
    ax.set_axis_off()
    ax.set_title(title)
    print(f'{title[0].upper() + title[1:]}: {(both[column] < 0).mean():.0%} of the squares burnt less in {LATER}, '
          f'{(both[column] > 0).mean():.0%} more.')
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap='RdBu_r'), ax=axes, shrink=0.6,
             label=f'burnt area {LATER} minus {EARLIER} (ha per km²)')
fig.suptitle(f'{AREA_LABEL}: change in burnt area, {EARLIER} to {LATER}')
fig.savefig(FIGS / '03_change_map.png', bbox_inches='tight', dpi=150)
plt.show()

# South to north: four bands with equal numbers of squares.
both['band'] = pd.qcut(both.to_crs(seasons[LATER]['crs']).centroid.y, 4,
                       labels=['south', 'south-centre', 'north-centre', 'north'])
sums = both.groupby('band', observed=True)[numbers].sum()
all_burns = {tag: sums[f'burnt_ha_{tag}'] + sums[f'burnt_unconfirmed_ha_{tag}'] for tag in ('earlier', 'later')}
bands = pd.DataFrame({
    f'burnt {EARLIER}, confirmed (ha)': sums['burnt_ha_earlier'],
    f'burnt {LATER}, confirmed (ha)': sums['burnt_ha_later'],
    'change, confirmed': sums['burnt_ha_later'] / sums['burnt_ha_earlier'] - 1,
    'change, including late burns': all_burns['later'] / all_burns['earlier'] - 1,
    f'late burns, share of {EARLIER}': sums['burnt_unconfirmed_ha_earlier'] / all_burns['earlier'],
    f'late burns, share of {LATER}': sums['burnt_unconfirmed_ha_later'] / all_burns['later'],
})
display(bands.round(3))

## 6. Outputs

In [ ]:
table.rename_axis('measure').reset_index().rename(columns={EARLIER: str(EARLIER), LATER: str(LATER)}).to_csv(
    PROCESSED / f'season_comparison_{AREA}.csv', index=False)
by_window.to_csv(PROCESSED / f'season_comparison_by_window_{AREA}.csv', index=False)
bands.to_csv(PROCESSED / f'season_comparison_by_band_{AREA}.csv')
print('Outputs written to', PROCESSED, 'and', FIGS)

### Caveats
- **The rule was checked on 2025 labels only.** The 2021 map uses the same thresholds without labels of its own, so its accuracy is unknown. Differences in smoke, cloud or soil moisture between the seasons could change how well the rule works.
- **Clear views.** Burns hidden by cloud or smoke, or ploughed in between clear views, are missed in both seasons; compare the mean clear share and the Sentinel-2 dates in the table before reading much into small differences.
- **Fire timing.** Fires that start and burn out between overpasses are missed by fire counts in both seasons. If burning has moved later in the day since 2021, counts would fall faster than burning. (A December 2025 iFOREST report, as reported by The Tribune and cited in the README, says most large Punjab farm fires in 2024 and 2025 occurred after 3 pm.) This comparison can show such a gap, but not its cause.
- **Overpass spacing.** NOAA-20 shifted in its orbit in April 2024 (CIRA, cited in the methodology), so even the same two satellites were not spaced the same way in both seasons.
- **One district.** Sangrur on its pre-2021 boundary (Malerkotla included) in both seasons.